# Logging my journey learning LLMs and PyTorch

It is long overdue for me to start learning PyTorch, and there is no better way to do it than building a toy LLM. I've gone through a list of papers that helped me understand how LLMs come to be, and I think it's a good idea to build a toy LLM to better connect the theory with practice.

# Starting the local colab docker image

Use the following command:
```
docker run -it --gpus=all -p 127.0.0.1:9000:8080 -p 127.0.0.1:6006:6006 --mount type=bind,src="${HOME}/colab/checkpoints",dst=/content/checkpoints us-docker.pkg.dev/colab-images/public/runtime
```

In [60]:
!pwd && ls

/content
checkpoints  sample_data  tiktoken


# Setup PyTorch

In [61]:
import torch

In [62]:
print(f"PyTorch Version: {torch.__version__}")
if (torch.cuda.is_available()):
    device = torch.device("cuda")
    print(f"Using GPU, CUDA version: {torch.version.cuda}, Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")

PyTorch Version: 2.10.0+cu128
Using GPU, CUDA version: 12.8, Number of GPUs: 1


# Get a working tokenizer
Let's use OpenAI's tiktoken tokenizer - alghough I am actually interested in building a minimum tokenizer from scratch later.

In [63]:
!git clone https://github.com/openai/tiktoken.git

fatal: destination path 'tiktoken' already exists and is not an empty directory.


In [64]:
import tiktoken

In [65]:
enc = tiktoken.get_encoding("r50k_base")

In [66]:
enc.encode("monkey d luffy")

[49572, 288, 300, 15352]

In [67]:
enc.decode(enc.encode("monkey d luffy"))

'monkey d luffy'

# Get the WebText dataset

In [68]:
from datasets import load_dataset

In [69]:
openwebtext = load_dataset("openwebtext")

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

In [70]:
openwebtext

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 8013769
    })
})

In [71]:
openwebtext['train']

Dataset({
    features: ['text'],
    num_rows: 8013769
})

## Test tokenizing the training data

In [72]:
enc.encode(openwebtext['train'][42]['text'])

[5159,
 8305,
 370,
 868,
 510,
 1165,
 1903,
 290,
 1719,
 2761,
 25446,
 736,
 284,
 3993,
 743,
 423,
 257,
 4633,
 2928,
 319,
 262,
 2612,
 11,
 257,
 2050,
 2523,
 198,
 198,
 8061,
 508,
 423,
 5876,
 38193,
 572,
 284,
 3993,
 743,
 307,
 379,
 3220,
 2526,
 286,
 2612,
 5287,
 11,
 4837,
 910,
 13,
 198,
 198,
 464,
 2050,
 11,
 3199,
 287,
 262,
 3427,
 8894,
 4913,
 11,
 3940,
 517,
 621,
 2026,
 11,
 830,
 661,
 329,
 1367,
 812,
 13,
 198,
 198,
 29193,
 1043,
 883,
 508,
 6989,
 1811,
 12513,
 286,
 3595,
 3993,
 547,
 517,
 1884,
 284,
 1205,
 262,
 4006,
 11,
 287,
 543,
 262,
 2612,
 10143,
 284,
 8901,
 6105,
 13,
 198,
 198,
 38897,
 910,
 2252,
 2267,
 318,
 2622,
 284,
 766,
 611,
 257,
 3092,
 286,
 3993,
 5640,
 2612,
 5287,
 393,
 262,
 2792,
 318,
 517,
 3716,
 13,
 198,
 198,
 1,
 42332,
 867,
 286,
 262,
 1243,
 326,
 4646,
 262,
 2863,
 286,
 2612,
 5287,
 635,
 4646,
 47104,
 26,
 922,
 5496,
 11,
 5517,
 11,
 3463,
 2994,
 290,
 407,
 9216,
 1583,
 5045,
 

# Embedding layer

In [73]:
vocabulary_size = enc.n_vocab
d_model = 768

embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)

Here we encode the phrase "monkey d luffy", which is then fed into the embedding_layer,
which produces d_model=768 embeddings, one 768 dimentional vector per token

In [74]:
embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

tensor([[[-0.8637,  0.8491, -0.0110,  ..., -0.7599, -0.0407, -1.1955],
         [ 0.5850,  0.8027, -0.9957,  ..., -0.5309, -0.8135, -0.9789],
         [-0.7814, -0.9618,  0.5772,  ..., -0.2213,  0.2149, -2.0650],
         [ 1.3312, -2.7489, -0.4639,  ..., -0.6890, -0.5125, -0.1494]]],
       grad_fn=<EmbeddingBackward0>)

In [75]:
embedding_layer.weight.shape

torch.Size([50257, 768])

Here we can see that the embedding layer is rather simple - it is a n_vocab x d_model matrix. So the embedding process is simply taking w[encoding].

## Positional Encoding

In [76]:
class LearnablePositionalEncoding(torch.nn.Module):
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        self.pos_embeddings = torch.nn.Embedding(max_seq_len, d_model)
    def forward(self, x):
        # x is of shape (batch_size, seq_len, d_model)
        seq_len = x.size(1)

        position_ids = torch.arange(seq_len, dtype=torch.long, device=x.device)

        pos_enc = self.pos_embeddings(position_ids)
        return x + pos_enc


In [77]:
pos_embed_layer = LearnablePositionalEncoding(1024, d_model)

In [78]:
pos_embed_layer(embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")])))

tensor([[[-1.5195,  1.5281,  1.2132,  ...,  1.3511, -0.5765, -0.9374],
         [ 0.8670,  0.4970, -0.8443,  ..., -0.3589, -0.2644,  0.1815],
         [-0.1699,  0.5233,  1.5322,  ..., -1.5655,  0.6914, -1.5121],
         [ 1.0629, -2.0695, -0.7549,  ..., -0.1003, -0.1859, -0.5342]]],
       grad_fn=<AddBackward0>)

# Decoder Only Transformer Layer

This is the key part of the transformer architecture. Each attention layer is made of multi-head attention, normalization, and position-wise feedforward.

In GPT-2, normalization is moved at the beginning of each layer.

In [79]:
embedding = embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

## Layer Normalization

https://arxiv.org/pdf/1607.06450

In [80]:
layer_norm = torch.nn.LayerNorm(d_model)

In [81]:
x = layer_norm(embedding)

In [82]:
x

tensor([[[-0.9133,  0.8430, -0.0389,  ..., -0.8069, -0.0694, -1.2536],
         [ 0.6618,  0.9007, -1.0729,  ..., -0.5628, -0.8729, -1.0544],
         [-0.7569, -0.9380,  0.6070,  ..., -0.1946,  0.2433, -2.0454],
         [ 1.3807, -2.7920, -0.4551,  ..., -0.6854, -0.5049, -0.1335]]],
       grad_fn=<NativeLayerNormBackward0>)

In [83]:
x.shape

torch.Size([1, 4, 768])

## Multi-Head Attention

In [84]:
n_head = 12

# project into Q, K, V
projection = torch.nn.Linear(d_model, d_model * 3)

In [85]:
projection(x).shape

torch.Size([1, 4, 2304])

In [86]:
proj = projection(x).view(1, -1, 12, d_model // n_head * 3).permute(0, 2, 1, 3)

In [87]:
q, k, v = proj.chunk(3, dim=3)

In [88]:
import math
atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_model)

In [89]:
mask = torch.tril(torch.ones_like(atten[0][0]))

In [90]:
atten = atten.masked_fill(mask == 0, -float('inf'))

In [91]:
atten

tensor([[[[-0.0440,    -inf,    -inf,    -inf],
          [-0.1279,  0.0659,    -inf,    -inf],
          [ 0.0225,  0.0029,  0.1238,    -inf],
          [ 0.0416,  0.0046, -0.0245,  0.1777]],

         [[ 0.0292,    -inf,    -inf,    -inf],
          [-0.0461,  0.0744,    -inf,    -inf],
          [ 0.0335, -0.1891, -0.0219,    -inf],
          [ 0.0232, -0.0416, -0.0896,  0.0615]],

         [[-0.1072,    -inf,    -inf,    -inf],
          [-0.0883, -0.0541,    -inf,    -inf],
          [-0.2245,  0.0110, -0.0242,    -inf],
          [-0.0400,  0.0514,  0.1845,  0.0730]],

         [[ 0.0062,    -inf,    -inf,    -inf],
          [-0.0261,  0.1976,    -inf,    -inf],
          [ 0.1720,  0.1017, -0.0367,    -inf],
          [-0.0564, -0.1375,  0.0288, -0.0099]],

         [[-0.0259,    -inf,    -inf,    -inf],
          [ 0.2406,  0.0775,    -inf,    -inf],
          [-0.0978,  0.0447,  0.2858,    -inf],
          [ 0.0797, -0.0351, -0.0170, -0.1876]],

         [[ 0.0006,    -inf,  

In [92]:
softmax = torch.nn.Softmax(dim=-1)

atten = softmax(atten)

In [93]:
atten

tensor([[[[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4517, 0.5483, 0.0000, 0.0000],
          [0.3239, 0.3176, 0.3584, 0.0000],
          [0.2472, 0.2382, 0.2314, 0.2832]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4699, 0.5301, 0.0000, 0.0000],
          [0.3641, 0.2914, 0.3445, 0.0000],
          [0.2584, 0.2422, 0.2309, 0.2685]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4915, 0.5085, 0.0000, 0.0000],
          [0.2868, 0.3629, 0.3504, 0.0000],
          [0.2239, 0.2453, 0.2802, 0.2506]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4443, 0.5557, 0.0000, 0.0000],
          [0.3645, 0.3397, 0.2958, 0.0000],
          [0.2464, 0.2272, 0.2683, 0.2581]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5407, 0.4593, 0.0000, 0.0000],
          [0.2762, 0.3185, 0.4053, 0.0000],
          [0.2805, 0.2501, 0.2547, 0.2147]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5165, 0.4835, 0.0000, 0.0000],
          [0.3711, 0.3

In [94]:
torch.matmul(atten, v).transpose(1, 2).reshape(1, 4, d_model)

tensor([[[-0.8240, -0.2384, -0.5516,  ..., -0.2459, -0.2610,  0.5373],
         [-0.2412, -0.4447, -0.4296,  ...,  0.4168, -0.7077, -0.0795],
         [-0.2602, -0.1211,  0.0570,  ...,  0.3921, -0.6340, -0.1487],
         [-0.1574, -0.1556, -0.2017,  ...,  0.1787, -0.5606, -0.2404]]],
       grad_fn=<UnsafeViewBackward0>)

### Putting it together

In [95]:
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # project into K Q V
        self.input_linear = torch.nn.Linear(d_model, d_model * 3)
    def forward(self, x):
        batch_size, sequence_length, _ = x.size()
        proj = self.input_linear(x).view(batch_size, sequence_length, self.n_heads, self.d_head * 3).permute(0, 2, 1, 3)
        q, k, v = proj.chunk(3, dim=-1)

        # Standard Transformer scaling: 1/sqrt(d_head)
        atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones_like(atten[0][0]))
        atten = atten.masked_fill(mask == 0, -float('inf'))
        atten = F.softmax(atten, dim=-1)

        return torch.matmul(atten, v).transpose(1, 2).reshape(batch_size, sequence_length, self.d_model)

In [96]:
multi_head_attention = MultiHeadAttention(d_model, n_head)
multi_head_attention(x)

tensor([[[-0.3583,  1.0040, -0.1103,  ..., -1.1566, -0.0920, -0.3014],
         [-0.5148,  0.9915, -0.0967,  ..., -0.7698,  0.0948,  0.0715],
         [-0.2781,  0.9261, -0.0508,  ..., -0.6812, -0.2727, -0.0969],
         [-0.0705,  0.8183,  0.0060,  ..., -0.4974, -0.1986, -0.1239]]],
       grad_fn=<UnsafeViewBackward0>)

## Position-wise Feed Forward

In [97]:
d_hidden = 3072

In [98]:
class DenseFeedForward(torch.nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(d_hidden, d_model)
        )
    def forward(self, x):
        return self.model(x)

In [99]:
y = multi_head_attention(x)

In [100]:
dff = DenseFeedForward(d_model, d_hidden)

In [101]:
dff(y)

tensor([[[ 0.0303,  0.0328,  0.2667,  ...,  0.0461, -0.2092,  0.1245],
         [-0.0244,  0.0954,  0.2770,  ...,  0.0398, -0.0872,  0.0623],
         [-0.0150,  0.0035,  0.2784,  ..., -0.0156, -0.0265, -0.0151],
         [-0.0134,  0.0030,  0.2088,  ...,  0.0320, -0.0135,  0.0077]]],
       grad_fn=<ViewBackward0>)

## The Full Transformer Layer

In [102]:
class TransformerDecoderOnly(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head):
        super().__init__()
        self.layer_norm_0 = torch.nn.LayerNorm(d_model)
        self.multi_head_attention = MultiHeadAttention(d_model, n_head)
        self.layer_norm_1 = torch.nn.LayerNorm(d_model)
        self.dff = DenseFeedForward(d_model, d_hidden)
    def forward(self, x):
        # layer normalization first
        x = self.layer_norm_0(x)

        # multi-head attention and residual
        identity = x
        x = self.multi_head_attention(x)
        x = x + identity

        # layer normalization before dense feed forward
        x = self.layer_norm_1(x)

        # dense feed forward and residual
        identity = x
        x = self.dff(x)
        return x + identity

In [103]:
transformer_decoder_only = TransformerDecoderOnly(d_model, d_hidden, n_head)

transformer_decoder_only(embedding)

tensor([[[-0.9051,  1.6003, -0.0359,  ..., -0.7015, -0.8205, -1.6651],
         [ 0.6170,  1.4842, -0.8813,  ..., -0.4447, -1.5132, -1.4508],
         [-0.7087, -0.6137,  0.8588,  ..., -0.4065,  0.0266, -2.7482],
         [ 1.2150, -2.1758, -0.4044,  ..., -0.6872, -0.4573, -0.4509]]],
       grad_fn=<AddBackward0>)

# Output Layer

The output layer is rather simple. matrix multiply with the input embedding matrix, and soft max. We can then reverse the tokenization.

In [104]:
torch.matmul(transformer_decoder_only(embedding), embedding_layer.weight.transpose(0, 1))

tensor([[[  4.0599,   9.0249,  -8.5997,  ..., -12.9227,   7.2632, -32.0714],
         [ 24.7106,  -6.6241,   9.6523,  ..., -38.0666, -48.7837, -48.8047],
         [ 25.4628,  15.4952,  -6.3944,  ...,  11.1534, -36.6881,  20.4650],
         [  8.3634,   6.0453,  34.3464,  ..., -53.3046, -26.2549,  -6.0751]]],
       grad_fn=<UnsafeViewBackward0>)

In [105]:
class Output(torch.nn.Module):
    def __init__(self, embedding_layer_weights):
        super().__init__()
        # Removed the extra LayerNorm here as it can skew initial logits
        self.embedding_layer_weights = embedding_layer_weights

    def forward(self, x):
        # x: (batch, seq, d_model)
        # weights: (vocab, d_model)
        return torch.matmul(x, self.embedding_layer_weights.transpose(0, 1))

# The full model

In [106]:
class ToyGPT(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head, vocab_size, layers, max_seq_len):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(vocab_size, d_model)
        self.pos_encoding_layer = LearnablePositionalEncoding(max_seq_len, d_model)
        self.transformers = torch.nn.Sequential(*[TransformerDecoderOnly(d_model, d_hidden, n_head) for _ in range(layers)])
        # Standard GPT-2: Final LayerNorm before the output head
        self.ln_f = torch.nn.LayerNorm(d_model)
        self.output_layer = Output(self.embedding_layer.weight)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, torch.nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, torch.nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.embedding_layer(x)
        x = self.pos_encoding_layer(x)
        x = self.transformers(x)
        x = self.ln_f(x) # Apply the missing final normalization
        return self.output_layer(x)

# Training

In [107]:
batch_size = 48
max_seq_len = 1024

In [108]:
enc._special_tokens

{'<|endoftext|>': 50256}

Pre-tokenize the training data to avoid CPU bottleneck.

In [109]:
from datasets import Dataset

rows = openwebtext["train"].num_rows
bos_token = '<|endoftext|>'

def tokenize(examples):
    out = enc.encode(examples['text']+ bos_token, allowed_special={bos_token})
    return {'tokenized_text': out}

tokenized_data = openwebtext["train"].map(tokenize, remove_columns=openwebtext["train"].column_names, num_proc=32)


In [110]:
def do_checkpoint(model, optimizer, running_encoding, row_number):
    torch.save({
        'row_number': row_number,
        'running_encoding': running_encoding,
        'optimizer_state_dict': optimizer.state_dict(),
        'model_state_dict': model.state_dict(),
    }, 'checkpoints/check_point_{}'.format(row_number))

In [111]:
@torch.no_grad()
def generate_with_sampling(model, prompt="The secret to life is", max_new_tokens=50, temperature=0.8, top_k=40):
    model.eval()
    input_ids = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        cond_ids = input_ids[:, -max_seq_len:]
        logits = model(cond_ids)
        logits = logits[:, -1, :] / temperature

        # Optional: Top-K filtering
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')

        # Sample from the distribution instead of taking the max
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat((input_ids, next_token), dim=1)
        if next_token.item() == enc._special_tokens[bos_token]:
            break

    out_text = enc.decode(input_ids[0].tolist())
    model.train()
    return out_text

# Try it with your current model scale
# print(generate_with_sampling(model, temperature=0.8, top_k=50))

In [112]:
import gc
import math
from tqdm.notebook import tqdm

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step > max_steps:
        return min_lr
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

@torch.no_grad()
def generate_sample(model, prompt="The secret to life is", max_new_tokens=50):
    model.eval()
    input_ids = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        cond_ids = input_ids[:, -max_seq_len:]
        logits = model(cond_ids)
        logits = logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat((input_ids, next_token), dim=1)
        if next_token.item() == enc._special_tokens[bos_token]:
            break
    out_text = enc.decode(input_ids[0].tolist())
    model.train()
    return out_text

def train(start_row_number = 0, check_point_interval = 100000, summary_writer = None):
    tokens_per_batch = (batch_size + 1) * max_seq_len // 2
    max_lr = 0.0006
    min_lr = max_lr * 0.1
    warmup_steps = 20000
    max_steps = rows

    encoding = []
    model = ToyGPT(d_model, d_hidden, n_head, enc.n_vocab, 12, max_seq_len)
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1)
    model.to(device)
    model.train()
    model = torch.compile(model)

    if start_row_number != 0:
        checkpoint = torch.load('checkpoints/check_point_{}'.format(start_row_number))
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        encoding = checkpoint['running_encoding']

    write_summary_threshold = start_row_number
    check_point_threshold = (start_row_number + check_point_interval) // check_point_interval * check_point_interval
    data = tokenized_data.select(range(start_row_number, tokenized_data.num_rows)).to_iterable_dataset(num_shards=8)

    for row in enumerate(tqdm(data, total=rows, initial=start_row_number)):
        cur_row_number = start_row_number + row[0]
        lr = get_lr(cur_row_number, warmup_steps, max_steps, max_lr, min_lr)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        encoding.extend([enc._special_tokens[bos_token]] + row[1]['tokenized_text'])

        if len(encoding) > tokens_per_batch:
            residual = encoding[tokens_per_batch:]
            encoding = encoding[0:tokens_per_batch + 1]
            batch = torch.empty((batch_size, max_seq_len), dtype=torch.long, device=device)
            target = torch.empty((batch_size, max_seq_len), dtype=torch.long, device=device)

            step_size = max_seq_len // 2
            for i in range(0, batch_size):
                start_idx = i * step_size
                batch[i] = torch.LongTensor(encoding[start_idx : start_idx + max_seq_len])
                target[i] = torch.LongTensor(encoding[start_idx + 1 : start_idx + max_seq_len + 1])

            optimizer.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                out = model(batch)
                loss = F.cross_entropy(out.view(-1, enc.n_vocab), target.view(-1))

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            encoding = residual

            if summary_writer is not None and cur_row_number >= write_summary_threshold:
                write_summary_threshold = (write_summary_threshold + 1000) // 1000 * 1000
                summary_writer.add_scalar('Loss/train', loss.item(), cur_row_number)
                summary_writer.add_scalar('LR/train', lr, cur_row_number)
                summary_writer.flush()

            if (cur_row_number + 1 >= check_point_threshold):
                print(f"\nCheckpointing at {cur_row_number + 1}. Loss: {loss.item():.4f}")
                do_checkpoint(model, optimizer, encoding, cur_row_number + 1)

                # Generate and log sample
                sample = generate_sample(model)
                print(f"Sample Generation: {sample}\n")
                sample_with_temperature = generate_with_sampling(model, temperature=0.8, top_k=50)
                print(f"Sample Generation with temperature 0.8: {sample_with_temperature}")
                if summary_writer is not None:
                    summary_writer.add_text('Generation/sample', sample, cur_row_number + 1)
                    summary_writer.add_text('Generation/sample_with_temp', sample_with_temperature, cur_row_number + 1)
                    summary_writer.flush()

                check_point_threshold = (cur_row_number + 1 + check_point_interval) // check_point_interval * check_point_interval

            gc.collect()

### Validation: Architecture and Initial Loss
For a randomly initialized model, the expected cross-entropy loss is $-\ln(1/V) = \ln(V)$, where $V$ is the vocabulary size. Let's verify this and check the output shapes.

In [113]:
import math

# 1. Check Output Shapes
test_model = ToyGPT(d_model, d_hidden, n_head, vocabulary_size, layers=1, max_seq_len=max_seq_len)
test_input = torch.randint(0, vocabulary_size, (1, 8)) # Batch size 1, Seq len 8
with torch.no_grad():
    logits = test_model(test_input)

print(f"Input shape: {test_input.shape}")
print(f"Output (logits) shape: {logits.shape} (Expected: [1, 8, {vocabulary_size}])")

# 2. Check Initial Loss
targets = torch.randint(0, vocabulary_size, (1, 8))
loss = F.cross_entropy(logits.view(-1, vocabulary_size), targets.view(-1))

theoretical_loss = math.log(vocabulary_size)
print(f"\nTheoretical Initial Loss: {theoretical_loss:.4f}")
print(f"Actual Initial Loss:      {loss.item():.4f}")
print(f"Difference:              {abs(loss.item() - theoretical_loss):.4f}")

if abs(loss.item() - theoretical_loss) < 0.5:
    print("\n✅ The initial loss is consistent with random initialization.")
else:
    print("\n⚠️ Initial loss is significantly different from theory. Check initialization or scaling.")

Input shape: torch.Size([1, 8])
Output (logits) shape: torch.Size([1, 8, 50257]) (Expected: [1, 8, 50257])

Theoretical Initial Loss: 10.8249
Actual Initial Loss:      10.9144
Difference:              0.0895

✅ The initial loss is consistent with random initialization.


In [114]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [115]:
from torch.utils.tensorboard import SummaryWriter

In [116]:
%tensorboard --logdir checkpoints/adamw_run_lr_0006 --bind_all


Reusing TensorBoard on port 6006 (pid 243504), started 16:03:27 ago. (Use '!kill 243504' to kill it.)

<IPython.core.display.Javascript object>

In [ ]:
summary_writer = SummaryWriter('./checkpoints/adamw_run_lr_0006')

train(start_row_number=7400004, check_point_interval=100000, summary_writer = summary_writer)